# 01 — Persiapan Data & Baseline BM25 (MIRACL-id)

**Penelitian:** *Comparative Evaluation of Fine-Tuned mBERT and mDeBERTa Cross-Encoders for Reranking on Indonesian MIRACL*
Juan Carlos Tepanus Pardosi (2506712092) — Advanced Information Retrieval, Fasilkom UI

Notebook ini adalah tahap pertama dari 4 notebook:

| Notebook | Isi | GPU? |
|---|---|---|
| **01_data_bm25** (ini) | Topics & qrels, split Train 90/10, BM25 top-100, hard negative (rasio 1:3), teks passage | Tidak perlu |
| 02_train_eval | Fine-tuning cross-encoder — Eksp. 1 (3 seed) & Eksp. 3 (25%/50% data) + efisiensi | L4 |
| 03_topk_efficiency | Eksp. 2: variasi top-k ∈ {20, 50, 100} dan trade-off akurasi vs latency | Tidak perlu |
| 04_analysis | Uji signifikansi, learning curve, truncation, analisis kualitatif | Tidak perlu |

**Hasil notebook ini (disimpan ke Google Drive):**
`topics.parquet`, `qrels.{train,dev}.tsv`, `splits.json`, `run.bm25.{train,dev}.top100.txt`,
`train_pairs.parquet`, `passages.parquet`, `bm25_dev_metrics.json`, `bm25_dev_per_query.csv`.

> 💡 Notebook ini **tidak butuh GPU** (BM25 berjalan di CPU via Pyserini/Java). Jalankan dengan runtime CPU agar kuota compute unit L4 tidak terpakai.

## 1. Instalasi Java 21 & Pyserini

In [1]:
# Pyserini (Anserini) membutuhkan Java 21
!apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-21-jdk-headless > /dev/null
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
!java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[0.019s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.019s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [2]:
!pip -q install pyserini pytrec-eval-terrier
# Jika muncul peringatan "restart session", lakukan Runtime > Restart session,
# lalu jalankan ulang sel Java di atas (tanpa apt-get) dan lanjutkan dari sel berikutnya.

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


### Environment Version

In [3]:
import sys, platform, importlib.metadata as md
print('Python      :', sys.version.split()[0], '|', platform.platform())
for pkg in ['pyserini', 'pytrec-eval-terrier', 'torch', 'transformers', 'sentencepiece',
            'numpy', 'pandas', 'scipy', 'scikit-learn', 'pyarrow', 'matplotlib']:
    try: print(f'{pkg:20s}:', md.version(pkg))
    except md.PackageNotFoundError: print(f'{pkg:20s}: (tidak terinstal)')
!java -version 2>&1 | head -1
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nvcc --version | tail -1
!lscpu | grep "Model name"; free -h | grep Mem

Python      : 3.13.18 | Linux-6.6.159+-x86_64-with-glibc2.39
pyserini            : 2.4.0
pytrec-eval-terrier : 0.5.11
torch               : 2.11.0+cpu
transformers        : 5.20.2
sentencepiece       : 0.2.2
numpy               : 2.1.3
pandas              : 2.2.3
scipy               : 1.16.3
scikit-learn        : 1.6.1
pyarrow             : 23.0.1
matplotlib          : 3.10.0
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
/bin/bash: line 1: nvidia-smi: command not found
/bin/bash: line 1: nvcc: command not found
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Mem:            12Gi       1.2Gi       6.6Gi       3.5Mi       5.1Gi        11Gi


## 2. Google Drive & direktori

In [3]:
# Mount Google Drive: semua data, skor, dan hasil disimpan di sini agar aman
# bila sesi Colab terputus.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

BASE_DIR    = '/content/drive/MyDrive/MIRACL_Reranking'   # ubah bila perlu
DATA_DIR    = f'{BASE_DIR}/data'      # topics, qrels, BM25 run, passages, training pairs
SCORES_DIR  = f'{BASE_DIR}/scores'    # skor cross-encoder per run (Dev top-100)
RESULTS_DIR = f'{BASE_DIR}/results'   # metrik (JSON), tabel (CSV), log
FIG_DIR     = f'{BASE_DIR}/figures'   # grafik (PNG)
for d in [DATA_DIR, SCORES_DIR, RESULTS_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)
print('BASE_DIR =', BASE_DIR)

BASE_DIR = /content/drive/MyDrive/MIRACL_Reranking


## 3. Konfigurasi

Semua nilai di bawah sesuai proposal (slide 8 & 11). `NEG_RATIO = 3` → setiap positif dipasangkan dengan 3 hard negative.

In [5]:
SEED          = 42
VAL_FRACTION  = 0.10            # 10% kueri Train untuk validasi (pemilihan checkpoint)
SUBSET_FRACS  = [0.25, 0.50]    # Eksp. 3 (nested: subset 25% ⊂ subset 50%)
BM25_DEPTH    = 100             # kandidat BM25 yang disimpan (k maksimum)
NEG_DEPTH     = 30              # hard negative tak berlabel diambil dari top-30 BM25
NEG_RATIO     = 3               # rasio negatif : positif = 3 : 1
INDEX_NAME    = 'miracl-v1.0-id'
TOPICS_TRAIN, TOPICS_DEV = 'miracl-v1.0-id-train', 'miracl-v1.0-id-dev'

## 4. Topics & qrels

Diambil dari Pyserini. Jika gagal (mis. nama topik berubah di versi baru), sel ini otomatis jatuh ke berkas resmi di Hugging Face `miracl/miracl`.
Angka yang diharapkan (paper MIRACL, Tabel 2): **Train 4.071 kueri / 41.358 judgments; Dev 960 kueri / 9.668 judgments**.

In [6]:
from collections import defaultdict

def _topics_pyserini(name):
    from pyserini.search import get_topics
    return {str(q): v['title'] for q, v in get_topics(name).items()}

def _qrels_pyserini(name):
    from pyserini.search import get_qrels_file
    return get_qrels_file(name)

def _hf_file(substr):
    """Unduh berkas dari repo HF miracl/miracl yang path-nya memuat `substr`."""
    from huggingface_hub import HfApi, hf_hub_download
    files = [f for f in HfApi().list_repo_files('miracl/miracl', repo_type='dataset')
             if substr in f]
    assert files, f'Tidak ditemukan berkas {substr} di miracl/miracl'
    return hf_hub_download('miracl/miracl', files[0], repo_type='dataset')

def load_topics(split):
    name = f'miracl-v1.0-id-{split}'
    try:
        return _topics_pyserini(name)
    except Exception as e:
        print(f'[fallback HF] topics {split}: {e}')
        path = _hf_file(f'topics.{name}.tsv')
        out = {}
        with open(path) as f:
            for line in f:
                qid, q = line.rstrip('\n').split('\t', 1)
                out[qid] = q
        return out

def load_qrels(split):
    name = f'miracl-v1.0-id-{split}'
    try:
        path = _qrels_pyserini(name)
    except Exception as e:
        print(f'[fallback HF] qrels {split}: {e}')
        path = _hf_file(f'qrels.{name}.tsv')
    qrels = defaultdict(dict)
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 4:
                qrels[str(p[0])][p[2]] = int(p[3])
    return dict(qrels), path

topics_train = load_topics('train')
topics_dev   = load_topics('dev')
qrels_train, qrels_train_path = load_qrels('train')
qrels_dev, qrels_dev_path     = load_qrels('dev')

def summarize(name, topics, qrels):
    n_j = sum(len(v) for v in qrels.values())
    n_pos = sum(r > 0 for v in qrels.values() for r in v.values())
    no_pos = sum(1 for q in topics if not any(r > 0 for r in qrels.get(q, {}).values()))
    print(f'{name}: {len(topics):,} kueri | {n_j:,} judgments | {n_pos:,} positif '
          f'({n_pos/len(topics):.2f}/kueri) | kueri tanpa positif: {no_pos}')

summarize('Train', topics_train, qrels_train)
summarize('Dev  ', topics_dev, qrels_dev)

Train: 4,071 kueri | 41,358 judgments | 12,505 positif (3.07/kueri) | kueri tanpa positif: 0
Dev  : 960 kueri | 9,668 judgments | 3,088 positif (3.22/kueri) | kueri tanpa positif: 0


## 5. Split Train → 90% training / 10% validasi (per kueri)

Split dilakukan **per kueri** (bukan per pasangan) agar kueri yang sama tidak muncul di training dan validasi. Subset Eksp. 3 diambil acak dari kueri training dengan seed tetap dan bersifat *nested* (25% ⊂ 50% ⊂ 100%); validasi 10% tetap sama untuk semua eksperimen. Dev resmi tidak disentuh selama training.

In [7]:
import numpy as np, json

all_train_qids = sorted(topics_train)            # urutan deterministik
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(all_train_qids))
n_val = int(round(VAL_FRACTION * len(all_train_qids)))
val_qids   = sorted(all_train_qids[i] for i in perm[:n_val])
train_qids = sorted(all_train_qids[i] for i in perm[n_val:])

rng_sub = np.random.RandomState(SEED)
shuffled_train = list(rng_sub.permutation(train_qids))
splits = {'train': train_qids, 'val': val_qids, 'dev': sorted(topics_dev)}
for frac in SUBSET_FRACS:
    n = int(round(frac * len(train_qids)))
    splits[f'train_{int(frac*100)}'] = sorted(shuffled_train[:n])

assert not set(train_qids) & set(val_qids)
assert set(splits['train_25']) <= set(splits['train_50']) <= set(train_qids)
with open(f'{DATA_DIR}/splits.json', 'w') as f:
    json.dump(splits, f)
print({k: len(v) for k, v in splits.items()})

{'train': 3664, 'val': 407, 'dev': 960, 'train_25': 916, 'train_50': 1832}


In [8]:
# Simpan topics & qrels ke Drive (dipakai notebook lain tanpa Pyserini)
import pandas as pd, shutil
topics_df = pd.concat([
    pd.DataFrame({'qid': list(topics_train), 'split': 'train', 'query': list(topics_train.values())}),
    pd.DataFrame({'qid': list(topics_dev), 'split': 'dev', 'query': list(topics_dev.values())}),
], ignore_index=True)
topics_df.to_parquet(f'{DATA_DIR}/topics.parquet', index=False)
shutil.copy(qrels_train_path, f'{DATA_DIR}/qrels.train.tsv')
shutil.copy(qrels_dev_path, f'{DATA_DIR}/qrels.dev.tsv')
topics_df.groupby('split')['query'].apply(lambda s: s.str.split().str.len().describe()).unstack().round(2)

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
dev,960.0,5.57,1.57,3.0,4.0,5.0,7.0,14.0
train,4071.0,5.48,1.55,2.0,4.0,5.0,6.0,14.0


## 6. First-stage BM25 (prebuilt index `miracl-v1.0-id`)

Konfigurasi sama dengan baseline resmi MIRACL di Pyserini (BM25 default k1=0.9, b=0.4, analyzer bahasa Indonesia). Index ±beberapa GB diunduh sekali. Diambil top-100 untuk seluruh kueri Train (sumber hard negative + kandidat validasi) dan Dev.

In [9]:
from pyserini.search.lucene import LuceneSearcher
from tqdm.auto import tqdm

searcher = LuceneSearcher.from_prebuilt_index(INDEX_NAME)
searcher.set_language('id')
print('Jumlah dokumen di index:', searcher.num_docs)

def bm25_search(topics, k=BM25_DEPTH, batch=512, threads=os.cpu_count()):
    qids, out = list(topics), {}
    for i in tqdm(range(0, len(qids), batch), desc='BM25'):
        b = qids[i:i + batch]
        hits = searcher.batch_search([topics[q] for q in b], b, k=k, threads=threads)
        for q in b:
            out[q] = [(h.docid, float(h.score)) for h in hits.get(q, [])]
    return out

def write_run(run, path, tag='bm25'):
    with open(path, 'w') as f:
        for q in sorted(run):
            for r, (d, s) in enumerate(run[q], 1):
                f.write(f'{q} Q0 {d} {r} {s:.6f} {tag}\n')

bm25_train = bm25_search(topics_train)
bm25_dev   = bm25_search(topics_dev)
write_run(bm25_train, f'{DATA_DIR}/run.bm25.train.top100.txt')
write_run(bm25_dev, f'{DATA_DIR}/run.bm25.dev.top100.txt')
print('Kueri tanpa hasil — train:', sum(not v for v in bm25_train.values()),
      '| dev:', sum(not v for v in bm25_dev.values()))

Jumlah dokumen di index: 1446315


BM25:   0%|          | 0/8 [00:00<?, ?it/s]

BM25:   0%|          | 0/2 [00:00<?, ?it/s]

Kueri tanpa hasil — train: 0 | dev: 0


## 7. Evaluasi baseline BM25 pada Dev (960 kueri)

Verifikasi ulang angka paper MIRACL: **nDCG@10 = 0,449; Recall@100 = 0,904**. Recall@20/50 adalah batas atas reranker untuk Eksp. 2.

In [10]:
import numpy as np
import pandas as pd
import pytrec_eval

def ranking_to_run(ranking):
    """{qid: [docid urut]} -> {qid: {docid: skor}} dengan skor menurun sesuai rank,
    sehingga pytrec_eval memakai urutan persis seperti daftar (tanpa efek tie)."""
    return {q: {d: float(len(docs) - i) for i, d in enumerate(docs)}
            for q, docs in ranking.items()}

def evaluate(ranking, qrels, recall_ks=(20, 50, 100)):
    """Hitung nDCG@10, MRR@10, MAP@100, Recall@k.

    ranking : {qid: [docid, ...]} terurut (maks. 100 dokumen)
    qrels   : {qid: {docid: rel}}
    Hanya kueri yang punya >=1 dokumen relevan yang dievaluasi; kueri yang
    tidak ada di ranking diberi skor 0 (bukan dibuang).
    Return: (rata-rata dict, DataFrame per kueri)
    """
    qids = sorted(q for q, rels in qrels.items() if any(r > 0 for r in rels.values()))
    q_sub = {q: qrels[q] for q in qids}
    full = ranking_to_run({q: ranking.get(q, []) for q in qids})
    top10 = ranking_to_run({q: ranking.get(q, [])[:10] for q in qids})
    # pytrec_eval melewati kueri dengan run kosong -> isi dummy agar tetap terhitung 0
    for run in (full, top10):
        for q in qids:
            if not run[q]:
                run[q] = {'__none__': 0.0}

    rec_spec = 'recall.' + ','.join(str(k) for k in recall_ks)
    ev_full = pytrec_eval.RelevanceEvaluator(q_sub, {'ndcg_cut.10', 'map_cut.100', rec_spec})
    ev_top10 = pytrec_eval.RelevanceEvaluator(q_sub, {'recip_rank'})
    r_full, r_top10 = ev_full.evaluate(full), ev_top10.evaluate(top10)

    rows = []
    for q in qids:
        row = {'qid': q,
               'ndcg@10': r_full[q]['ndcg_cut_10'],
               'mrr@10': r_top10[q]['recip_rank'],
               'map@100': r_full[q]['map_cut_100']}
        for k in recall_ks:
            row[f'recall@{k}'] = r_full[q][f'recall_{k}']
        rows.append(row)
    per_q = pd.DataFrame(rows).set_index('qid')
    return per_q.mean().to_dict(), per_q

In [11]:
bm25_rank_dev = {q: [d for d, _ in v] for q, v in bm25_dev.items()}
bm25_mean, bm25_perq = evaluate(bm25_rank_dev, qrels_dev)
print({k: round(v, 4) for k, v in bm25_mean.items()})
print(f"Selisih vs paper: nDCG@10 {bm25_mean['ndcg@10'] - 0.449:+.4f} | "
      f"Recall@100 {bm25_mean['recall@100'] - 0.904:+.4f}")

bm25_perq.to_csv(f'{RESULTS_DIR}/bm25_dev_per_query.csv')
with open(f'{RESULTS_DIR}/bm25_dev_metrics.json', 'w') as f:
    json.dump({'n_queries': len(bm25_perq), **bm25_mean}, f, indent=2)

# Validasi juga BM25 pada 10% validasi (dipakai sebagai acuan di notebook 02)
bm25_val_mean, _ = evaluate({q: [d for d, _ in bm25_train[q]] for q in val_qids},
                            {q: qrels_train[q] for q in val_qids if q in qrels_train})
print('BM25 pada validasi:', {k: round(v, 4) for k, v in bm25_val_mean.items()})

{'ndcg@10': 0.4486, 'mrr@10': 0.5306, 'map@100': 0.3893, 'recall@20': 0.6788, 'recall@50': 0.8295, 'recall@100': 0.9041}
Selisih vs paper: nDCG@10 -0.0004 | Recall@100 +0.0001
BM25 pada validasi: {'ndcg@10': 0.4753, 'mrr@10': 0.5563, 'map@100': 0.4108, 'recall@20': 0.7231, 'recall@50': 0.8793, 'recall@100': 0.9384}


## 8. Pasangan latih (q, p, y) dengan hard negative (rasio 1:3)

Untuk setiap kueri training:
- **Positif (y=1):** semua passage dengan `rel = 1` di qrels Train.
- **Negatif (y=0):** sebanyak `3 × jumlah positif`, diambil dengan prioritas:
  1. passage `rel = 0` di qrels (negatif terverifikasi anotator; di MIRACL ini sudah *hard* karena berasal dari pool retriever),
  2. lalu passage **tak berlabel** dari top-30 BM25 (risiko *false negative*, sehingga hanya dipakai sebagai pelengkap).
- Kueri tanpa positif dilewati. Sampling dilakukan **sekali** (seed 42) sehingga semua model & seed memakai pasangan yang identik.

In [12]:
def build_pairs(qids, qrels, bm25_run, neg_ratio=NEG_RATIO, neg_depth=NEG_DEPTH, seed=SEED):
    rng = np.random.RandomState(seed)
    rows, stats = [], defaultdict(int)
    for q in qids:
        judged = qrels.get(q, {})
        pos = sorted(d for d, r in judged.items() if r > 0)
        if not pos:
            stats['skipped_no_pos'] += 1
            continue
        judged_neg = sorted(d for d, r in judged.items() if r <= 0)
        unlabeled  = [d for d, _ in bm25_run.get(q, [])[:neg_depth] if d not in judged]
        rng.shuffle(judged_neg)
        rng.shuffle(unlabeled)
        n_neg = neg_ratio * len(pos)
        negs = [(d, 'judged') for d in judged_neg[:n_neg]]
        negs += [(d, 'bm25_unlabeled') for d in unlabeled[:n_neg - len(negs)]]
        if len(negs) < n_neg:
            stats['queries_short_of_negatives'] += 1
        rows += [(q, d, 1, 'positive') for d in pos]
        rows += [(q, d, 0, src) for d, src in negs]
    df = pd.DataFrame(rows, columns=['qid', 'docid', 'label', 'source'])
    return df, dict(stats)

pairs, pair_stats = build_pairs(train_qids, qrels_train, bm25_train)
for frac in SUBSET_FRACS:
    key = f'train_{int(frac*100)}'
    pairs[f'in_{int(frac*100)}'] = pairs.qid.isin(set(splits[key]))
pairs.to_parquet(f'{DATA_DIR}/train_pairs.parquet', index=False)

n_pos, n_neg = (pairs.label == 1).sum(), (pairs.label == 0).sum()
print(f'Total pasangan: {len(pairs):,} | positif {n_pos:,} | negatif {n_neg:,} '
      f'| rasio neg:pos aktual = {n_neg/n_pos:.2f}')
print('Statistik:', pair_stats)
print(pairs.source.value_counts().to_string())
for frac in SUBSET_FRACS:
    col = f'in_{int(frac*100)}'
    print(f'Subset {int(frac*100)}%: {pairs[col].sum():,} pasangan dari {pairs[pairs[col]].qid.nunique():,} kueri')
with open(f'{RESULTS_DIR}/train_pairs_stats.json', 'w') as f:
    json.dump({'n_pairs': int(len(pairs)), 'n_pos': int(n_pos), 'n_neg': int(n_neg),
               'neg_ratio_actual': float(n_neg / n_pos), 'neg_ratio_target': NEG_RATIO,
               'by_source': pairs.source.value_counts().to_dict(), **pair_stats}, f, indent=2)

Total pasangan: 44,814 | positif 11,244 | negatif 33,570 | rasio neg:pos aktual = 2.99
Statistik: {'queries_short_of_negatives': 41}
source
judged            19431
bm25_unlabeled    14139
positive          11244
Subset 25%: 11,318 pasangan dari 916 kueri
Subset 50%: 22,454 pasangan dari 1,832 kueri


## 9. Teks passage

Teks diambil dari index Lucene (field *raw*: `title` + `text`). Hanya passage yang dibutuhkan yang disimpan: pasangan latih, kandidat top-100 BM25 untuk validasi & Dev, serta passage relevan Dev (untuk analisis kualitatif). Bila index tidak menyimpan teks, sel ini memakai korpus resmi `miracl/miracl-corpus` dari Hugging Face.

In [13]:
needed = set(pairs.docid)
for q in val_qids:
    needed.update(d for d, _ in bm25_train[q])
for q in topics_dev:
    needed.update(d for d, _ in bm25_dev[q])
    needed.update(d for d, r in qrels_dev.get(q, {}).items() if r > 0)
needed = sorted(needed)
print(f'Passage yang dibutuhkan: {len(needed):,}')

def parse_doc(doc):
    raw = doc.raw() if doc is not None else None
    if raw is None:
        return None
    try:
        j = json.loads(raw)
        title = j.get('title', '') or ''
        text = j.get('text', j.get('contents', '')) or ''
    except json.JSONDecodeError:
        title, text = '', raw
    return title, text

passages, missing = {}, []
for i in tqdm(range(0, len(needed), 5000), desc='Ambil passage'):
    chunk = needed[i:i + 5000]
    docs = searcher.batch_doc(chunk, threads=os.cpu_count())
    for d in chunk:
        parsed = parse_doc(docs.get(d))
        if parsed is None:
            missing.append(d)
        else:
            passages[d] = parsed
print(f'Berhasil: {len(passages):,} | tidak ada teks: {len(missing):,}')

Passage yang dibutuhkan: 125,805


Ambil passage:   0%|          | 0/26 [00:00<?, ?it/s]

Berhasil: 125,805 | tidak ada teks: 0


In [14]:
# Fallback: ambil teks dari korpus resmi di Hugging Face bila ada passage tanpa teks
if missing:
    import gzip
    from huggingface_hub import HfApi, hf_hub_download
    files = [f for f in HfApi().list_repo_files('miracl/miracl-corpus', repo_type='dataset')
             if f.startswith('miracl-corpus-v1.0-id/')]
    todo = set(missing)
    for fname in tqdm(files, desc='HF corpus'):
        p = hf_hub_download('miracl/miracl-corpus', fname, repo_type='dataset')
        opener = gzip.open if p.endswith('.gz') else open
        with opener(p, 'rt') as f:
            for line in f:
                j = json.loads(line)
                if j['docid'] in todo:
                    passages[j['docid']] = (j.get('title', ''), j.get('text', ''))
                    todo.discard(j['docid'])
        if not todo:
            break
    print('Masih hilang setelah fallback:', len(todo))

In [15]:
pass_df = pd.DataFrame([(d, t, x) for d, (t, x) in passages.items()],
                       columns=['docid', 'title', 'text'])
pass_df.to_parquet(f'{DATA_DIR}/passages.parquet', index=False)
lens = (pass_df.title + ' ' + pass_df.text).str.split().str.len()
print(f'Tersimpan {len(pass_df):,} passage. Panjang (kata): median {lens.median():.0f}, '
      f'p95 {lens.quantile(.95):.0f}, maks {lens.max()}')
pass_df.sample(3, random_state=0)

Tersimpan 125,805 passage. Panjang (kata): median 65, p95 165, maks 1862


,docid,title,text
80188,336509#0,"Talok, Turen, Malang",Talok adalah sebuah desa di wilayah Kecamatan ...
27995,16848#2,Final Fantasy,"Terhitung Juni 2006, di Jepang telah dirilis d..."
105237,62435#5,Hamengkubawana X,"Sebagai Gubernur, ia tidak menguber penghargaa..."


## 10. Ringkasan

Jika semua sel berhasil, lanjutkan ke **02_train_eval.ipynb** dengan runtime **L4 GPU**.
Periksa bahwa nDCG@10 BM25 ≈ 0,449 dan Recall@100 ≈ 0,904 (selisih kecil wajar karena versi Pyserini/Lucene).

In [16]:
!ls -lh {DATA_DIR} {RESULTS_DIR}

/content/drive/MyDrive/MIRACL_Reranking/data:
total 57M
-rw------- 1 root root  39M Oct 26 08:10 passages.parquet
-rw------- 1 root root 173K Oct 26 08:08 qrels.dev.tsv
-rw------- 1 root root 741K Oct 26 08:08 qrels.train.tsv
-rw------- 1 root root 3.2M Oct 26 08:10 run.bm25.dev.top100.txt
-rw------- 1 root root  14M Oct 26 08:10 run.bm25.train.top100.txt
-rw------- 1 root root  60K Oct 26 08:08 splits.json
-rw------- 1 root root 151K Oct 26 08:08 topics.parquet
-rw------- 1 root root 357K Oct 26 08:10 train_pairs.parquet

/content/drive/MyDrive/MIRACL_Reranking/results:
total 56K
-rw------- 1 root root 225 Oct 26 08:10 bm25_dev_metrics.json
-rw------- 1 root root 55K Oct 26 08:10 bm25_dev_per_query.csv
-rw------- 1 root root 255 Oct 26 08:10 train_pairs_stats.json
